<a href="https://colab.research.google.com/github/abduyea/Career-Trends-Analyzer/blob/main/notebooks/Proejct_setup.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Abdulfetah Adem

Spencer K

#Project Objective

Develop a data-driven tool that identifies the most in-demand technical and professional skills using real-world job posting data.

The tool will highlight skill gaps and provide an interactive dashboard to explore trends across industries, regions, and time periods.

# DATA LOADING

In [1]:
# import libreries
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import drive
from os.path import join


In [11]:
# Mount Google Drive
drive.mount("/content/drive", force_remount=True)


Mounted at /content/drive


In [12]:
# define project path and directories
ROOT_DIR = "/content/drive/MyDrive/Career-Trends-Analyzer"
RAW_DIR = os.path.join(ROOT_DIR, "data", "raw")

COMP_DIR = os.path.join(RAW_DIR, "companies")
JOBS_DIR = os.path.join(RAW_DIR, "jobs")
MAP_DIR = os.path.join(RAW_DIR, "mappings")
POSTINGS_CSV = os.path.join(RAW_DIR, "postings.csv")


In [13]:
# Load all CSV files inside a subfolder
def load_csvs(folder, prefix):
    return {
        f"{prefix}_{f[:-4]}": pd.read_csv(os.path.join(folder, f))
        for f in os.listdir(folder)
        if f.lower().endswith(".csv")
    }


In [14]:
#Load ALL Data
datasets = {}

# 1 posting file
datasets["postings"] = pd.read_csv(POSTINGS_CSV)

# Subfolder CSVs
datasets.update(load_csvs(COMP_DIR, "companies"))
datasets.update(load_csvs(JOBS_DIR, "jobs"))
datasets.update(load_csvs(MAP_DIR, "mappings"))

list(datasets.keys())


['postings',
 'companies_employee_counts',
 'companies_company_specialities',
 'companies_companies',
 'companies_company_industries',
 'jobs_salaries',
 'jobs_job_industries',
 'jobs_job_skills',
 'jobs_benefits',
 'mappings_industries',
 'mappings_skills']

# PREPROCESSING (BUILD MASTER TABLE)

In [15]:
# Build Master Table
def build_master_table(datasets):
    df = datasets["postings"].copy()

    # Merge company info
    if "companies_companies" in datasets:
        df = df.merge(
            datasets["companies_companies"], on="company_id", how="left"
        )

    # Merge industries
    if "jobs_job_industries" in datasets and "mappings_industries" in datasets:
        df = (
            df.merge(datasets["jobs_job_industries"], on="job_id", how="left")
            .merge(datasets["mappings_industries"], on="industry_id", how="left")
        )

    # Merge skills (skill_abr)
    if "jobs_job_skills" in datasets and "mappings_skills" in datasets:
        df = (
            df.merge(datasets["jobs_job_skills"], on="job_id", how="left")
            .merge(datasets["mappings_skills"], on="skill_abr", how="left")
        )

    return df


In [16]:
# create master table
master = build_master_table(datasets)
master.shape


(280665, 44)

In [17]:
master.head()

,job_id,company_name,title,description_x,max_salary,pay_period,location,company_id,views,med_salary,...,state,country,city,zip_code_y,address,url,industry_id,industry_name,skill_abr,skill_name
0,921716,Corcoran Sawyer Smith,Marketing Coordinator,Job descriptionA leading real estate firm in N...,20.0,HOURLY,"Princeton, NJ",2774458.0,20.0,NaN,...,NJ,US,Jersey City,07302,242 Tenth Street,https://www.linkedin.com/company/corcoran-sawy...,44.0,Real Estate,MRKT,Marketing
1,921716,Corcoran Sawyer Smith,Marketing Coordinator,Job descriptionA leading real estate firm in N...,20.0,HOURLY,"Princeton, NJ",2774458.0,20.0,NaN,...,NJ,US,Jersey City,07302,242 Tenth Street,https://www.linkedin.com/company/corcoran-sawy...,44.0,Real Estate,SALE,Sales
2,1829192,NaN,Mental Health Therapist/Counselor,"At Aspen Therapy and Wellness , we are committ...",50.0,HOURLY,"Fort Collins, CO",NaN,1.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,HCPR,Health Care Provider
3,10998357,The National Exemplar,Assitant Restaurant Manager,The National Exemplar is accepting application...,65000.0,YEARLY,"Cincinnati, OH",64896719.0,8.0,NaN,...,Ohio,US,Mariemont,45227,6880 Wooster Pike,https://www.linkedin.com/company/the-national-...,32.0,Restaurants,MGMT,Management
4,10998357,The National Exemplar,Assitant Restaurant Manager,The National Exemplar is accepting application...,65000.0,YEARLY,"Cincinnati, OH",64896719.0,8.0,NaN,...,Ohio,US,Mariemont,45227,6880 Wooster Pike,https://www.linkedin.com/company/the-national-...,32.0,Restaurants,MNFC,Manufacturing


In [7]:
# Project folders
ROOT_DIR = "/content/drive/MyDrive/Career-Trends-Analyzer"
DATA_DIR = os.path.join(ROOT_DIR, "data")
RAW_DIR = os.path.join(DATA_DIR, "raw")

COMP_DIR = os.path.join(RAW_DIR, "companies")
JOBS_DIR = os.path.join(RAW_DIR, "jobs")
MAP_DIR = os.path.join(RAW_DIR, "mappings")

for n, p in {
    "RAW_DIR": RAW_DIR,
    "COMP_DIR": COMP_DIR,
    "JOBS_DIR": JOBS_DIR,
    "MAP_DIR": MAP_DIR,
}.items():
    print(f"{n}: {p}")


RAW_DIR: /content/drive/MyDrive/Career-Trends-Analyzer/data/raw
COMP_DIR: /content/drive/MyDrive/Career-Trends-Analyzer/data/raw/companies
JOBS_DIR: /content/drive/MyDrive/Career-Trends-Analyzer/data/raw/jobs
MAP_DIR: /content/drive/MyDrive/Career-Trends-Analyzer/data/raw/mappings


In [5]:
# Load all CSV files in a folder
def load_csvs(folder: str, prefix: str):
    items = {}
    if not os.path.isdir(folder):
        return items

    for f in os.listdir(folder):
        if f.lower().endswith(".csv"):
            path = os.path.join(folder, f)
            key = f"{prefix}_{f[:-4]}"
            items[key] = pd.read_csv(path)
    return items


In [8]:
# Load datasets from all raw folders
datasets = {}

# Top-level postings.csv
postings_path = os.path.join(RAW_DIR, "postings.csv")
if os.path.isfile(postings_path):
    datasets["postings"] = pd.read_csv(postings_path)

# Subfolders
datasets.update(load_csvs(COMP_DIR, "companies"))
datasets.update(load_csvs(JOBS_DIR, "jobs"))
datasets.update(load_csvs(MAP_DIR, "mappings"))

print("Loaded datasets:", list(datasets.keys()))


Loaded datasets: ['postings', 'companies_employee_counts', 'companies_company_specialities', 'companies_companies', 'companies_company_industries', 'jobs_salaries', 'jobs_job_industries', 'jobs_job_skills', 'jobs_benefits', 'mappings_industries', 'mappings_skills']


In [ ]:
# View all dataset shapes
for name, df in datasets.items():
    print(f"{name}: {df.shape}")


postings: (123849, 31)
companies_employee_counts: (35787, 4)
companies_company_specialities: (169387, 2)
companies_companies: (24473, 10)
companies_company_industries: (24375, 2)
jobs_salaries: (40785, 8)
jobs_job_industries: (164808, 2)
jobs_job_skills: (213768, 2)
jobs_benefits: (67943, 3)
mappings_industries: (422, 2)
mappings_skills: (35, 2)


In [ ]:
# Set project and data paths
ROOT_DIR = "/content/drive/MyDrive/Career-Trends-Analyzer"
DATA_DIR = os.path.join(ROOT_DIR, "data")
RAW_DIR = os.path.join(DATA_DIR, "raw")

for name, path in {
    "ROOT_DIR": ROOT_DIR,
    "DATA_DIR": DATA_DIR,
    "RAW_DIR": RAW_DIR,
}.items():
    print(f"{name}: {path}")


ROOT_DIR: /content/drive/MyDrive/Career-Trends-Analyzer
DATA_DIR: /content/drive/MyDrive/Career-Trends-Analyzer/data
RAW_DIR: /content/drive/MyDrive/Career-Trends-Analyzer/data/raw


In [ ]:
# Set project directories
import os

ROOT_DIR = "/content/drive/MyDrive/Career-Trends-Analyzer"
DATA_DIR = os.path.join(ROOT_DIR, "data")

print("DATA_DIR:", DATA_DIR)


DATA_DIR: /content/drive/MyDrive/Career-Trends-Analyzer/data


In [ ]:
# List all CSV files in the data folder
csv_files = [
    f for f in os.listdir(DATA_DIR)
    if f.lower().endswith(".csv")
]

print("Found CSV files:")
for f in csv_files:
    print("•", f)


Found CSV files:


In [ ]:
# Load all CSV datasets
import pandas as pd

datasets = {
    f.replace(".csv", ""): pd.read_csv(os.path.join(DATA_DIR, f))
    for f in csv_files
}


In [ ]:
# Print dataset shapes
for name, df in datasets.items():
    print(f"{name}: {df.shape}")


In [ ]:
datasets.keys()


dict_keys([])

In [ ]:
# project path
ROOT_DIR = "/content/drive/MyDrive/Career-Trends-Analyzer"

DATA_DIR   = join(ROOT_DIR, "data")
NOTE_DIR   = join(ROOT_DIR, "notebooks")
SRC_DIR    = join(ROOT_DIR, "src")
MODEL_DIR  = join(ROOT_DIR, "model")
REPORT_DIR = join(ROOT_DIR, "report")

for name, path in {
    "ROOT_DIR": ROOT_DIR,
    "DATA_DIR": DATA_DIR,
    "NOTE_DIR": NOTE_DIR,
    "SRC_DIR": SRC_DIR,
    "MODEL_DIR": MODEL_DIR,
    "REPORT_DIR": REPORT_DIR,
}.items():
    print(f"{name}: {path}")


ROOT_DIR: /content/drive/MyDrive/Career-Trends-Analyzer
DATA_DIR: /content/drive/MyDrive/Career-Trends-Analyzer/data
NOTE_DIR: /content/drive/MyDrive/Career-Trends-Analyzer/notebooks
SRC_DIR: /content/drive/MyDrive/Career-Trends-Analyzer/src
MODEL_DIR: /content/drive/MyDrive/Career-Trends-Analyzer/model
REPORT_DIR: /content/drive/MyDrive/Career-Trends-Analyzer/report


In [ ]:
# Define project path for git hub
BASE_DIR = "/content/drive/MyDrive/Carer_trend_analayzer_project"
REPO_DIR = os.path.join(BASE_DIR, "Career-Trends-Analyzer")
GIT_URL = "https://github.com/abduyea/Career-Trends-Analyzer.git"

os.makedirs(BASE_DIR, exist_ok=True)
%cd $BASE_DIR


/content/drive/MyDrive/Carer_trend_analayzer_project


### Clone github repo to google collab

In [ ]:
# Navigate into your project folder
%cd /content/drive/MyDrive/Carer_trend_analayzer_project
# CLONE your GitHub repo into this folder
!git clone https://github.com/abduyea/Career-Trends-Analyzer.git



In [ ]:
# navigate in claoned repo
%cd /content/drive/MyDrive/Carer_trend_analayzer_project/Career-Trends-Analyzer


/content/drive/MyDrive/Carer_trend_analayzer_project/Career-Trends-Analyzer


In [ ]:
! git pull

Already up to date.
